# Real-Market MSA-Delta Backtest

This notebook runs the market-signal adjusted delta hedge on real Yahoo Finance data. The underlying is SPY. VIX is used as the historical option-implied volatility signal, and the signal set also includes realized-volatility acceleration, SPY volume shock, SPY drawdown, XLK sector divergence, and 13-week Treasury-bill rate changes.

The committed figures and table below are from a run over Yahoo Finance data from **2019-01-01 through 2024-12-31**, with an out-of-sample hedge test from **2022-12-23 through 2024-12-27** across **24** monthly at-the-money European-call episodes.


## Reproduce the Run

The next cell downloads data, processes signals, fits the risk-state model on the training sample, runs the hedge backtest, and saves tables/figures. It requires network access to Yahoo Finance.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from examples.run_real_market_pipeline import run_real_market_pipeline, save_outputs

result = run_real_market_pipeline(start="2019-01-01", end="2025-01-01")
save_outputs(result)
result.hedging_metrics.round(6)


## Out-of-Sample Hedging Metrics

| Strategy | Episodes | MAE | RMSE | 95% Loss ES | 99% Loss ES | Avg Cost | Avg Turnover |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| FV-BS | 24 | 1.7066 | 2.4268 | 5.6774 | 5.7238 | 0.6214 | 1242.8932 |
| Rolling-BS | 24 | 1.8540 | 2.4408 | 5.1898 | 5.2029 | 0.6219 | 1243.7169 |
| VIX-BS | 24 | 1.5804 | 2.0461 | 3.8183 | 3.8377 | 0.5726 | 1145.1539 |
| MSA-Delta | 24 | 1.8441 | 2.3693 | 5.0746 | 5.2029 | 0.6159 | 1231.7708 |

Lower MAE/RMSE/Expected Shortfall is better. All strategies receive the same initial option premium inside each episode, so these numbers isolate hedge performance rather than option sale price differences.


## Risk-State Distribution

| Risk State | Trading Dates |
| --- | ---: |
| normal | 446 |
| elevated | 54 |
| stress | 7 |

The MSA-Delta rule classifies most test dates as normal, with relatively sparse elevated and stress dates. That is intentional: the hedge should only become more aggressive when the signal stack indicates unusual market risk.


## Visual Diagnostics

### SPY and VIX

![SPY and VIX](../docs/figures/real_market_spy_vix.png)

### MSA-Delta Risk Score and States

![Risk states](../docs/figures/real_market_risk_states.png)

### Hedging Metrics

![Hedging metrics](../docs/figures/real_market_hedging_metrics.png)

### Episode Hedging Errors

![Episode errors](../docs/figures/real_market_episode_errors.png)


## Analysis

- **FV-BS** is the fixed-volatility benchmark. It has MAE of 1.7066 and RMSE of 2.4268 in the out-of-sample period.
- **Rolling-BS** updates realized volatility through time, but in this run it does not improve average hedging error relative to FV-BS.
- **VIX-BS** uses VIX directly as a point-in-time option-implied volatility input. It performs best in this run, with the lowest MAE, RMSE, transaction cost, turnover, and tail-loss expected shortfall.
- **MSA-Delta** uses a risk-state rule rather than VIX directly as the hedge volatility. It improves RMSE and tail-risk metrics versus FV-BS, but its MAE is slightly worse than FV-BS and worse than VIX-BS. This means the current risk-state rule helps in some large-error episodes but needs validation tuning if the objective is average absolute hedging error.


## Conclusion

The real-market experiment supports the main research direction: point-in-time market signals, especially option-implied volatility, contain useful information for hedging beyond fixed historical volatility. On this SPY/VIX sample, the pure VIX-based hedge is the strongest simple benchmark. The MSA-Delta rule reduces RMSE and tail loss relative to FV-BS, but it does not yet dominate on MAE. The next research step is to tune the MSA-Delta weights, quantile thresholds, and volatility multipliers on a validation window, then test once on a held-out period. A stronger version should also use option-chain implied-volatility skew and term structure rather than VIX alone.
